# 06 · CatBoost

Gradient boosting con árboles simétricos (todos los nodos de un mismo nivel usan la misma división) y ordenamiento aleatorio de los datos, lo que reduce el sobreajuste.

Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos). Las corridas van al experimento `bogoprice-catboost` de MLflow.

In [ ]:
import catboost as cb
import pandas as pd

from bogoprice.experimentos import evaluar, preparar_datos
from bogoprice.tracking import iniciar_mlflow

# Tag de autor de MLflow
AUTOR = "Edgar"
iniciar_mlflow("bogoprice-catboost")
datos = preparar_datos()
resultados = []


def catb(**p):
    # Valores base: cada prueba cambia solo uno de ellos
    base = {"loss_function": "RMSE", "iterations": 1500, "learning_rate": 0.05, "depth": 8,
            "l2_leaf_reg": 3, "random_strength": 1, "subsample": 0.8,
            "border_count": 254, "rsm": 1.0}
    modelo = cb.CatBoostRegressor(verbose=0, random_seed=0, thread_count=-1,
                                  allow_writing_files=False, **(base | p))
    return modelo, base | p

## 1. Conjuntos de variables

La configuración base con los tres conjuntos de variables.

In [ ]:
for version in ["v1", "v2", "v2_geo"]:
    modelo, params = catb()
    m = evaluar(datos, f"catboost_{version}", modelo, version, {"modelo": "CatBoost", **params}, AUTOR)[0]
    resultados.append({"corrida": f"catboost_{version}", "conjunto": version,
                       "parametro": "base", "valor": "-", **m})
    print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

## 2. Un hiperparámetro a la vez (v2 con y sin coordenadas)

Se parte de la configuración base y se prueban **4 valores de cada hiperparámetro**, dejando los demás constantes.

| Hiperparámetro | Qué controla | Base | Valores probados |
|---|---|---|---|
| `loss_function` | Función de pérdida que se minimiza | `RMSE` (cuadrática) | `MAE`, `Huber`, `LogCosh`, `Lq` (q = 1,5) |
| `iterations` | Número de árboles | 1500 | 500, 1000, 2500, 4000 |
| `learning_rate` | Cuánto corrige cada árbol | 0,05 | 0,02, 0,03, 0,08, 0,15 |
| `depth` | Profundidad de cada árbol | 8 | 4, 6, 7, 10 |
| `l2_leaf_reg` | Regularización L2 de las hojas | 3 | 1, 5, 10, 30 |
| `random_strength` | Aleatoriedad al elegir divisiones | 1 | 0, 0,5, 2, 5 |
| `subsample` | Fracción de anuncios usada en cada árbol | 0,8 | 0,5, 0,6, 0,7, 1,0 |
| `border_count` | Cortes posibles por variable numérica | 254 | 32, 64, 128, 512 |
| `rsm` | Fracción de variables evaluadas en cada división | 1,0 | 0,3, 0,5, 0,7, 0,9 |

In [ ]:
# Cada parámetro se prueba con 4 valores; los demás quedan en la base
pruebas = {
    "loss_function": ["MAE", "Huber:delta=0.5", "LogCosh", "Lq:q=1.5"],
    "iterations": [500, 1000, 2500, 4000],
    "learning_rate": [0.02, 0.03, 0.08, 0.15],
    "depth": [4, 6, 7, 10],
    "l2_leaf_reg": [1, 5, 10, 30],
    "random_strength": [0, 0.5, 2, 5],
    "subsample": [0.5, 0.6, 0.7, 1.0],
    "border_count": [32, 64, 128, 512],
    "rsm": [0.3, 0.5, 0.7, 0.9],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            modelo, params = catb(**{parametro: valor})
            nombre = f"catboost_{parametro}{valor}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                        {"modelo": "CatBoost", "parametro_variado": parametro, **params}, AUTOR)[0]
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})
        print(f"{version:7s} {parametro} listo")

## Resumen

MdAPE en anuncios nuevos de cada prueba, con y sin coordenadas. La fila `base` es la referencia.

In [ ]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto", values="mdape_nuevos").round(2)

**Efecto de cada hiperparámetro:** diferencia entre el mejor y el peor de sus 4 valores, y mejora del mejor valor frente a la base (positivo = mejora). Los dos con mayor mejora son los candidatos para explorar en detalle.

In [ ]:
base = tabla[tabla["parametro"] == "base"].set_index("conjunto")["mdape_nuevos"]
pruebas_df = tabla[tabla["parametro"] != "base"]
efecto = (pruebas_df.groupby(["conjunto", "parametro"])["mdape_nuevos"]
          .agg(mejor="min", peor="max")
          .assign(rango=lambda d: d["peor"] - d["mejor"]))
efecto["mejora_vs_base"] = base.reindex(efecto.index.get_level_values("conjunto")).values - efecto["mejor"]
efecto.round(2).sort_values(["conjunto", "mejora_vs_base"], ascending=[True, False])

## Conclusiones

- _(qué hiperparámetros mejoran frente a la base y cuáles no)_
- _(los dos con mayor efecto, que se explorarán en detalle)_
- _(efecto de las coordenadas)_